In [2]:
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options


import pandas as pd

In [3]:
login_url = "https://www.tilastopaja.info/login.php"
target_url = "https://www.tilastopaja.info/db/alltfull.php?Ind=0&Event=390&Sex=1&area=&All=0&Age=99"

options = Options()

options.page_load_strategy = 'normal'

driver = webdriver.Chrome(options=options)
driver.get(login_url)


The chromedriver version (134.0.6998.165) detected in PATH at /opt/homebrew/bin/chromedriver might not be compatible with the detected chrome version (135.0.7049.41); currently, chromedriver 135.0.7049.42 is recommended for chrome 135.*, so it is advised to delete the driver in PATH and retry


In [4]:
# Find input fields and login
driver.find_element(By.NAME, "user").send_keys(os.environ["TILASTOPAJA_USER"])
driver.find_element(By.NAME, "password").send_keys(os.environ["TILASTOPAJA_PASS"] + Keys.RETURN)
driver.find_element(By.XPATH,"//input[@type='button' and @value='Login']").click()

## Getting all event Names and IDs

In [5]:
events = []
EventDatabase = pd.DataFrame(columns=['Event Name','Event ID',"Indoor"])
idx=0
for io in ["0","1"]:
    url = "https://www.tilastopaja.info/db/alltfull.php?Ind="+io+"&Event=390&Sex=1&area=&All=0&Age=99"
    # print("checking : {}".format(url))
    driver.get(url)
    soup = BeautifulSoup(driver.page_source,"html.parser")
    for option in soup.find_all('div',class_="info-left")[0].find_all('option'):
        if idx==0:
            idx+=1
            continue
        event_name = option.text.strip()
        try:
            event_id = int(option['value'].split("&")[1].split('=')[1])
        except:
            continue
        # if event_id not in EventDatabase['Event ID'].to_list():
        if len(EventDatabase[(EventDatabase['Event ID']==event_id)&(EventDatabase['Indoor']==io)])==0:
            EventDatabase.loc[len(EventDatabase),['Event Name','Event ID',"Indoor"]] =  [event_name,event_id,io]
            idx+=1
            print("{} - {} - {}".format(event_name,event_id,"indoor" if io == "0" else "outdoor"))

100m - 40 - indoor
200m - 50 - indoor
300m - 60 - indoor
400m - 70 - indoor
600m - 80 - indoor
800m - 90 - indoor
1000m - 100 - indoor
1500m - 110 - indoor
One Mile - 120 - indoor
2000m - 130 - indoor
3000m - 140 - indoor
5000m - 160 - indoor
10,000m - 170 - indoor
5km, Road - 185 - indoor
10km, Road - 186 - indoor
Half Marathon - 190 - indoor
Marathon - 200 - indoor
3000m Steeplechase - 220 - indoor
110m Hurdles - 270 - indoor
400m Hurdles - 300 - indoor
High Jump - 310 - indoor
Pole Vault - 320 - indoor
Long Jump - 330 - indoor
Triple Jump - 340 - indoor
Shot Put - 350 - indoor
Discus Throw - 360 - indoor
Hammer Throw - 380 - indoor
Javelin Throw - 390 - indoor
Javelin Throw (Old Model) - 391 - indoor
Decathlon - 410 - indoor
20,000m Race Walk - 450 - indoor
10km Race Walk - 490 - indoor
20km Race Walk - 500 - indoor
35km Race Walk - 530 - indoor
50km Race Walk - 550 - indoor
4 x 100m - 560 - indoor
4 x 200m - 570 - indoor
4 x 400m - 580 - indoor
4 x 800m - 590 - indoor
4 x 1500m - 6

In [6]:
EventDatabase

,Event Name,Event ID,Indoor
0,100m,40,0
1,200m,50,0
2,300m,60,0
3,400m,70,0
4,600m,80,0
...,...,...,...
67,Pentathlon,399,1
68,3000 m walk,420,1
69,5000 m walk,430,1
70,10000 m walk,440,1


In [7]:
EventDatabase.to_hdf("/Users/mariograndi/Desktop/Sports Data/Data/Events.h5","df",mode='w',index=False)

/var/folders/v5/h8qjlslj4x9dg0j_3kwds6940000gn/T/ipykernel_99749/2464383125.py:1: FutureWarning: Starting with pandas version 3.0 all arguments of to_hdf except for the argument 'path_or_buf' will be keyword-only.
  EventDatabase.to_hdf("/Users/mariograndi/Desktop/Sports Data/Data/Events.h5","df",mode='w',index=False)
/var/folders/v5/h8qjlslj4x9dg0j_3kwds6940000gn/T/ipykernel_99749/2464383125.py:1: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['Event Name', 'Event ID', 'Indoor'], dtype='object')]

  EventDatabase.to_hdf("/Users/mariograndi/Desktop/Sports Data/Data/Events.h5","df",mode='w',index=False)


# Find Athletes for each event

In [19]:
event_data = {}
all_athletes = pd.DataFrame(columns=['PID','Athlete Name',"Athlete Country","Athlete Sex","Athlete URL"])
for sex in ["1","2"]:
    for io in ["0","1"]:
        for event_name, event_id,nul in EventDatabase.loc[EventDatabase["Indoor"]==io].values:
            event_url  = "https://www.tilastopaja.info/db/alltfull.php?Ind="+io+"&Event="+str(event_id)+"&Sex="+sex+"&area=&All=0&Age=99"

            print("Scraping {} | {} | {} --> from {}".format(event_name,"male" if sex=="1" else "female","outdoor" if io=="1" else "indoor",event_url))

            data = []
            datum=[]


            driver.get(event_url)
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "table")))

            soup = BeautifulSoup(driver.page_source,"html.parser")

            table = soup.find_all('table')[-1]

            rows = table.find_all("tr")

            for row in rows:
                columns = row.find_all("td")
                datum = [col.text.strip() for col in columns]
                if datum[0] != '':
                    try:
                        int(datum[0])
                        col = columns[4]
                        link = col.find("a")["href"]
                        id = link.split("ID=")[1]
                        name =  col.find("a").text
                        
                        datum[4] = name
                        datum.append(io)
                        datum.append(sex)    
                        datum.append(int(id))
                        datum.append(link)
                        data.append(datum)
                    except:
                        break
                        
            event_data[(event_name,io,sex)] = pd.DataFrame(data,columns=["Rank","Score","Wind","Record","Athlete Name","Athlete Country",'Athlete Date of Birth','Position','Event',"Event Location",'Date of Event',"Indoor",'Athlete Sex','PID','Athlete URL'])
            
            all_athletes = pd.concat([all_athletes,event_data[(event_name,io,sex)].loc[~event_data[(event_name,io,sex)]['PID'].duplicated(),['PID','Athlete Name',"Athlete Country","Athlete Sex","Athlete URL"]]],ignore_index=True)

            all_athletes = all_athletes.loc[~all_athletes['PID'].duplicated()]

            io_text = "indoor" if io == "1" else "outdoor"

            all_athletes.loc[all_athletes['PID'].isin(list(event_data[(event_name,io,sex)]['PID'].drop_duplicates())),[io_text+"_"+event_name]] = 1
all_athletes.fillna(0,inplace=True)

Scraping 100m | male | indoor --> from https://www.tilastopaja.info/db/alltfull.php?Ind=0&Event=40&Sex=1&area=&All=0&Age=99
Scraping 200m | male | indoor --> from https://www.tilastopaja.info/db/alltfull.php?Ind=0&Event=50&Sex=1&area=&All=0&Age=99
Scraping 300m | male | indoor --> from https://www.tilastopaja.info/db/alltfull.php?Ind=0&Event=60&Sex=1&area=&All=0&Age=99
Scraping 400m | male | indoor --> from https://www.tilastopaja.info/db/alltfull.php?Ind=0&Event=70&Sex=1&area=&All=0&Age=99
Scraping 600m | male | indoor --> from https://www.tilastopaja.info/db/alltfull.php?Ind=0&Event=80&Sex=1&area=&All=0&Age=99
Scraping 800m | male | indoor --> from https://www.tilastopaja.info/db/alltfull.php?Ind=0&Event=90&Sex=1&area=&All=0&Age=99
Scraping 1000m | male | indoor --> from https://www.tilastopaja.info/db/alltfull.php?Ind=0&Event=100&Sex=1&area=&All=0&Age=99
Scraping 1500m | male | indoor --> from https://www.tilastopaja.info/db/alltfull.php?Ind=0&Event=110&Sex=1&area=&All=0&Age=99
Scra

In [20]:
all_athletes

,PID,Athlete Name,Athlete Country,Athlete Sex,Athlete URL,outdoor_100m,outdoor_200m,outdoor_300m,outdoor_400m,outdoor_600m,...,indoor_Long Jump,indoor_Triple Jump,indoor_Shot Put,indoor_Weight Throw,indoor_Heptathlon,indoor_Pentathlon,indoor_3000 m walk,indoor_5000 m walk,indoor_10000 m walk,indoor_4 x 400 m
0,45032,Usain Bolt,JAM,1,https://www.tilastopaja.info/db/at.php?Sex=1&I...,1.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,45173,Tyson Gay,USA,1,https://www.tilastopaja.info/db/at.php?Sex=1&I...,1.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,69837,Yohan Blake,JAM,1,https://www.tilastopaja.info/db/at.php?Sex=1&I...,1.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4109,Asafa Powell,JAM,1,https://www.tilastopaja.info/db/at.php?Sex=1&I...,1.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,23308,Justin Gatlin,USA,1,https://www.tilastopaja.info/db/at.php?Sex=1&I...,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70439,14173,,MIX,2,https://www.tilastopaja.info/db/at.php?Sex=2&I...,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
70440,26933,,MIX,2,https://www.tilastopaja.info/db/at.php?Sex=2&I...,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
70441,14328,,MIX,2,https://www.tilastopaja.info/db/at.php?Sex=2&I...,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
70444,71865,,USA,2,https://www.tilastopaja.info/db/at.php?Sex=2&I...,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [23]:
for key,top_performances in event_data.items():
    event_name = key[0]
    location = "outdoor" if key[1]=="0" else 'indoor'
    sex = "male" if key[2]=="1" else 'female'
    
    directory="/Users/mariograndi/Desktop/Sports Data/Data/Events/"
    file_name = "top_performance_"+event_name+"_"+location+"_"+sex+".h5"
    print("Writing : {}".format(file_name))
    top_performances.to_hdf(directory+file_name,'df',mode='w',index=False)
all_athletes.to_hdf("/Users/mariograndi/Desktop/Sports Data/Data/Roster/all_athletes_roster.h5","df",mode='w',index=False)

Writing : top_performance_100m_outdoor_male.h5
Writing : top_performance_200m_outdoor_male.h5
Writing : top_performance_300m_outdoor_male.h5
Writing : top_performance_400m_outdoor_male.h5
Writing : top_performance_600m_outdoor_male.h5
Writing : top_performance_800m_outdoor_male.h5
Writing : top_performance_1000m_outdoor_male.h5
Writing : top_performance_1500m_outdoor_male.h5
Writing : top_performance_One Mile_outdoor_male.h5
Writing : top_performance_2000m_outdoor_male.h5
Writing : top_performance_3000m_outdoor_male.h5
Writing : top_performance_5000m_outdoor_male.h5
Writing : top_performance_10,000m_outdoor_male.h5
Writing : top_performance_5km, Road_outdoor_male.h5
Writing : top_performance_10km, Road_outdoor_male.h5
Writing : top_performance_Half Marathon_outdoor_male.h5
Writing : top_performance_Marathon_outdoor_male.h5
Writing : top_performance_2000m Steeplechase_outdoor_male.h5
Writing : top_performance_3000m Steeplechase_outdoor_male.h5
Writing : top_performance_110m Hurdles_outd

/var/folders/v5/h8qjlslj4x9dg0j_3kwds6940000gn/T/ipykernel_1071/3777919757.py:9: FutureWarning: Starting with pandas version 3.0 all arguments of to_hdf except for the argument 'path_or_buf' will be keyword-only.
  top_performances.to_hdf(directory+file_name,'df',mode='w',index=False)


Writing : top_performance_High Jump_outdoor_male.h5
Writing : top_performance_Pole Vault_outdoor_male.h5
Writing : top_performance_Long Jump_outdoor_male.h5
Writing : top_performance_Triple Jump_outdoor_male.h5
Writing : top_performance_Shot Put_outdoor_male.h5
Writing : top_performance_Discus Throw_outdoor_male.h5
Writing : top_performance_Hammer Throw_outdoor_male.h5
Writing : top_performance_Javelin Throw_outdoor_male.h5
Writing : top_performance_Javelin Throw (Old Model)_outdoor_male.h5
Writing : top_performance_Decathlon_outdoor_male.h5
Writing : top_performance_5000m Race Walk_outdoor_male.h5
Writing : top_performance_20,000m Race Walk_outdoor_male.h5
Writing : top_performance_20km Race Walk_outdoor_male.h5
Writing : top_performance_35km Race Walk_outdoor_male.h5
Writing : top_performance_50km Race Walk_outdoor_male.h5
Writing : top_performance_4 x 100m_outdoor_male.h5
Writing : top_performance_4 x 200m_outdoor_male.h5
Writing : top_performance_4 x 400m_outdoor_male.h5
Writing : 

/var/folders/v5/h8qjlslj4x9dg0j_3kwds6940000gn/T/ipykernel_1071/3777919757.py:10: FutureWarning: Starting with pandas version 3.0 all arguments of to_hdf except for the argument 'path_or_buf' will be keyword-only.
  all_athletes.to_hdf("/Users/mariograndi/Desktop/Sports Data/Data/Roster/all_athletes_roster.h5","df",mode='w',index=False)
/var/folders/v5/h8qjlslj4x9dg0j_3kwds6940000gn/T/ipykernel_1071/3777919757.py:10: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block1_values] [items->Index(['PID', 'Athlete Name', 'Athlete Country', 'Athlete Sex', 'Athlete URL'], dtype='object')]

  all_athletes.to_hdf("/Users/mariograndi/Desktop/Sports Data/Data/Roster/all_athletes_roster.h5","df",mode='w',index=False)


In [25]:
all_athletes

,PID,Athlete Name,Athlete Country,Athlete Sex,Athlete URL,outdoor_100m,outdoor_200m,outdoor_300m,outdoor_400m,outdoor_600m,...,indoor_Long Jump,indoor_Triple Jump,indoor_Shot Put,indoor_Weight Throw,indoor_Heptathlon,indoor_Pentathlon,indoor_3000 m walk,indoor_5000 m walk,indoor_10000 m walk,indoor_4 x 400 m
0,45032,Usain Bolt,JAM,1,https://www.tilastopaja.info/db/at.php?Sex=1&I...,1.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,45173,Tyson Gay,USA,1,https://www.tilastopaja.info/db/at.php?Sex=1&I...,1.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,69837,Yohan Blake,JAM,1,https://www.tilastopaja.info/db/at.php?Sex=1&I...,1.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4109,Asafa Powell,JAM,1,https://www.tilastopaja.info/db/at.php?Sex=1&I...,1.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,23308,Justin Gatlin,USA,1,https://www.tilastopaja.info/db/at.php?Sex=1&I...,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70439,14173,,MIX,2,https://www.tilastopaja.info/db/at.php?Sex=2&I...,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
70440,26933,,MIX,2,https://www.tilastopaja.info/db/at.php?Sex=2&I...,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
70441,14328,,MIX,2,https://www.tilastopaja.info/db/at.php?Sex=2&I...,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
70444,71865,,USA,2,https://www.tilastopaja.info/db/at.php?Sex=2&I...,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
